# Agricultural Field Aggregation into Voronoi Boundaries — Google Colab

Reads GeoParquet field polygon files from a Google Drive shared folder, filters by field probability, and aggregates field count and total area into Voronoi administrative boundaries.

**Run the cells in order.** The first two cells install packages and mount Drive — both require a browser authentication step.

## 1. Install Dependencies

In [ ]:
%%capture
!pip install geopandas pyarrow pyogrio

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Parameters & Paths

In [ ]:
from pathlib import Path
from datetime import datetime

# --- Input: folder containing .parquet files (searched recursively) ---
RAW_DATA_DIR = Path("/content/drive/MozFields/Vector_Data")

# --- Input: Voronoi boundaries ---
VORONOI_PATH = Path("/content/drive/AgCAP/data/processed/input_voronoi/vor_poly_20260417.gpkg")

# --- Output (timestamp appended to filename) ---
_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_PATH = Path(f"/out/voronoi_aggregated_{_timestamp}.gpkg")

# --- Field probability filter ---
# Polygons with rf_v7_prb below this threshold are excluded before analysis.
FIELD_PROB_THRESHOLD = 65

# --- Test mode ---
# When True, only the first TEST_NUMBER parquet files are processed.
TEST_MODE   = False
TEST_NUMBER = 5

## 4. Imports & Setup

In [ ]:
import gc
import warnings

import geopandas as gpd
import pandas as pd

warnings.filterwarnings("ignore")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
print(f"Output will be saved to: {OUTPUT_PATH}")

## 5. Load Voronoi Boundaries

The Voronoi layer is loaded once and kept in memory. A synthetic integer ID is assigned from the row index so the join works regardless of whether the layer has a unique identifier column.

In [ ]:
voronoi = gpd.read_file(VORONOI_PATH)

# Assign a synthetic integer ID from the row index so the join always works,
# regardless of whether the layer has a meaningful unique identifier column.
VORONOI_ID_COL = "_vor_id"
voronoi[VORONOI_ID_COL] = voronoi.index

# Reproject to a projected CRS if not already projected
if not voronoi.crs.is_projected:
    voronoi = voronoi.to_crs("EPSG:32736")  # UTM Zone 36S

METRIC_CRS = voronoi.crs

print(f"Voronoi CRS  : {voronoi.crs}")
print(f"Voronoi rows : {len(voronoi):,}")
print(f"Voronoi cols : {list(voronoi.columns)}")
voronoi.head(2)

## 6. Discover Input Files

Searches the input folder and all its subfolders recursively for `.parquet` files.

In [ ]:
all_files = sorted(RAW_DATA_DIR.rglob("*.parquet"))

if TEST_MODE:
    files_to_process = all_files[:TEST_NUMBER]
    print(f"TEST_MODE ON — processing {len(files_to_process)} of {len(all_files)} files")
else:
    files_to_process = all_files
    print(f"Processing all {len(files_to_process)} files")

for f in files_to_process:
    print(" ", f.relative_to(RAW_DATA_DIR))

## 7. Iterative Processing

For each parquet file the pipeline is:

1. **Load** the GeoParquet file.
2. **Filter** out polygons whose `rf_v7_prb` (field probability) is below `FIELD_PROB_THRESHOLD`.
3. **Reproject** to the Voronoi's projected CRS and **calculate polygon area** in hectares.
4. **Convert geometries to centroids** — makes the spatial join a fast point-in-polygon lookup.
5. **Spatial join** centroids → Voronoi polygons (both in the same CRS).
6. **Group by** Voronoi ID to get per-cell field count and total area for this file.
7. **Accumulate** results into a global dictionary with `+=`.
8. **Free memory** before loading the next file.

In [ ]:
# Global accumulator: maps voronoi pcode -> {"count": int, "area_ha": float}
global_tally: dict[str, dict] = {}

for i, fpath in enumerate(files_to_process, start=1):
    print(f"[{i}/{len(files_to_process)}] {fpath.name} ...", end=" ", flush=True)

    # --- Load ---
    fields = gpd.read_parquet(fpath)

    # --- Filter by field probability ---
    before = len(fields)
    fields = fields[fields["rf_v7_prb"] >= FIELD_PROB_THRESHOLD].copy()
    print(f"({len(fields):,}/{before:,} kept) ...", end=" ", flush=True)

    if fields.empty:
        print("skipped")
        del fields
        gc.collect()
        continue

    # --- Area calculation in metric CRS ---
    fields_metric = fields.to_crs(METRIC_CRS)
    fields_metric["area_ha"] = fields_metric.geometry.area / 10_000  # m² → ha

    # --- Convert to centroids (already in metric CRS, ready for the join) ---
    fields_metric["geometry"] = fields_metric.geometry.centroid
    centroids = fields_metric[["area_ha", "geometry"]]

    del fields, fields_metric
    gc.collect()

    # --- Spatial join: point-in-polygon ---
    joined = gpd.sjoin(
        centroids,
        voronoi[[VORONOI_ID_COL, "geometry"]],
        how="inner",
        predicate="within",
    )

    # --- Local aggregation per Voronoi cell ---
    local_agg = (
        joined.groupby(VORONOI_ID_COL)["area_ha"]
        .agg(count="count", area_ha="sum")
    )

    # --- Accumulate into global tally ---
    for vor_id, row in local_agg.iterrows():
        if vor_id not in global_tally:
            global_tally[vor_id] = {"count": 0, "area_ha": 0.0}
        global_tally[vor_id]["count"]   += int(row["count"])
        global_tally[vor_id]["area_ha"] += float(row["area_ha"])

    print(f"{int(local_agg['count'].sum()):,} fields joined")

    del centroids, joined, local_agg
    gc.collect()

print(f"\nDone. Voronoi cells with at least one field: {len(global_tally):,}")

## 8. Build Output GeoDataFrame

Convert the global tally into a DataFrame and join it back onto the Voronoi layer. Cells with no matched fields receive 0.

In [ ]:
tally_df = pd.DataFrame.from_dict(global_tally, orient="index")
tally_df.index.name = VORONOI_ID_COL
tally_df = tally_df.reset_index().rename(columns={"count": "field_count", "area_ha": "total_area_ha"})
tally_df["field_count"]   = tally_df["field_count"].astype(int)
tally_df["total_area_ha"] = tally_df["total_area_ha"].astype(float)

result = voronoi.merge(tally_df, on=VORONOI_ID_COL, how="left")
result["field_count"]   = result["field_count"].fillna(0).astype(int)
result["total_area_ha"] = result["total_area_ha"].fillna(0.0).astype(float)

print(f"Output rows          : {len(result):,}")
print(f"Cells with fields    : {(result['field_count'] > 0).sum():,}")
print(f"Total fields counted : {result['field_count'].sum():,}")
print(f"Total area (ha)      : {result['total_area_ha'].sum():,.1f}")
result.head()

## 9. Export

Saves the result to the Colab local filesystem. Download it from the file browser on the left sidebar, or run the optional cell below to copy it back to Drive.

In [ ]:
result.to_file(OUTPUT_PATH, driver="GPKG")
print(f"Saved to: {OUTPUT_PATH}")

In [ ]:
# Optional: copy output back to Google Drive
import shutil

DRIVE_OUTPUT_PATH = Path("/content/drive/MozFields") / OUTPUT_PATH.name
shutil.copy(OUTPUT_PATH, DRIVE_OUTPUT_PATH)
print(f"Copied to Drive: {DRIVE_OUTPUT_PATH}")